# Análise de Plots — Séries Temporais de Receita

Exploração visual de cada série temporal do experimento.

| Seção | Conteúdo |
|---|---|
| 1. Resumo Geral | Tabela de todas as séries com métricas e previsões |
| 2. Galeria de Plots | Plots individuais (PNG gerados pelo experimento) com cartão de métricas |
| 3. ACF/PACF | Correlograma do treino de cada série (autocorrelação e autocorrelação parcial) |
| 4. Análise por Grupo de Sinal | Desempenho segmentado por tipo de sinal |
| 5. Distribuição de Métricas | Boxplot de RMSE/MAPE por modelo |
| 6. Análise de Série Individual | Inspeção detalhada de uma série escolhida |

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from IPython.display import display, Image, HTML

plt.rcParams.update({
    'figure.dpi'        : 120,
    'font.family'       : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.4,
})

from config import PLOTS_DIR, RESULTS_DIR, CORES, COR_MODELO

C = CORES
print('Configuração OK')
print(f'  PLOTS_DIR   : {PLOTS_DIR}')
print(f'  RESULTS_DIR : {RESULTS_DIR}')

In [ ]:
from utils.preprocessing import (
    carregar_dados, construir_series, mapear_nomes,
    filtrar_series, preparar_serie, dividir_serie,
)

df_raw        = carregar_dados()
series_todas  = construir_series(df_raw)
nomes_map     = mapear_nomes(df_raw)
series_validas, excluidas = filtrar_series(series_todas)

print(f'Total de séries no dataset : {len(series_todas)}')
print(f'Séries válidas (filtradas)  : {len(series_validas)}')
print(f'Séries excluídas            : {len(excluidas)}')

In [ ]:
# Carrega os CSVs de resultado mais recentes
metricas_files  = sorted(RESULTS_DIR.glob('metricas_*.csv'))
previsoes_files = sorted(RESULTS_DIR.glob('previsoes_*.csv'))

if not metricas_files or not previsoes_files:
    print('AVISO: Execute experiment.py ou 01_experimento_geral.ipynb primeiro.')
    metricas_df = previsoes_df = None
else:
    metricas_df  = pd.read_csv(metricas_files[-1])
    previsoes_df = pd.read_csv(previsoes_files[-1])
    print(f'Métricas  : {metricas_files[-1].name}  ({len(metricas_df)} registros)')
    print(f'Previsões : {previsoes_files[-1].name}  ({len(previsoes_df)} séries)')
    print(f'Colunas (previsões): {list(previsoes_df.columns)}')

---
## 1. Resumo Geral

In [ ]:
if previsoes_df is not None:
    # Detecta colunas de previsão disponíveis
    cols_fmt  = [c for c in previsoes_df.columns if 'fmt' in c.lower()]
    cols_var  = [c for c in previsoes_df.columns if 'var' in c.lower() or 'Var' in c]
    cols_base = ['Codigo', 'Nome', 'Grupo_Sinal', 'Melhor_Modelo', 'RMSE_Teste', 'MAPE_Teste_Pct']
    cols_base = [c for c in cols_base if c in previsoes_df.columns]
    cols_show = cols_base + cols_fmt + cols_var

    fmt_dict = {}
    if 'RMSE_Teste'     in previsoes_df.columns: fmt_dict['RMSE_Teste']     = '{:,.0f}'
    if 'MAPE_Teste_Pct' in previsoes_df.columns: fmt_dict['MAPE_Teste_Pct'] = '{:.2f}%'
    for c in cols_var:
        fmt_dict[c] = '{:+.2f}%'

    # Cores por modelo vencedor
    def _cor_modelo(val):
        return f'color: {COR_MODELO.get(val, "#333")}; font-weight: bold'

    styled = (
        previsoes_df[cols_show]
        .style
        .format(fmt_dict, na_rep='-')
        .hide(axis='index')
    )
    if 'Melhor_Modelo' in previsoes_df.columns:
        styled = styled.applymap(_cor_modelo, subset=['Melhor_Modelo'])
    display(styled)
else:
    # Fallback: mostra lista de séries válidas
    info = [{'Codigo': k, 'Nome': nomes_map.get(k, 'N/D'), 'N_obs': len(v)}
            for k, v in series_validas.items()]
    display(pd.DataFrame(info).style.hide(axis='index'))

In [ ]:
# Exibe o plot resumo de modelos gerado pelo experimento (se disponível)
resumo_path = PLOTS_DIR / 'resumo_modelos.png'
if resumo_path.exists():
    display(HTML('<h4>Resumo de Desempenho dos Modelos (gerado pelo experimento)</h4>'))
    display(Image(str(resumo_path), width=850))
else:
    print('Plot resumo_modelos.png não encontrado — execute o experimento primeiro.')

In [ ]:
if previsoes_df is not None and 'Melhor_Modelo' in previsoes_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Contagem de vencedores
    ax = axes[0]
    venc = previsoes_df['Melhor_Modelo'].value_counts()
    bars = ax.bar(venc.index, venc.values,
                  color=[COR_MODELO.get(m, C['azul_med']) for m in venc.index],
                  edgecolor='white', width=0.5)
    for bar, val in zip(bars, venc.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.1, str(val),
                ha='center', fontweight='bold')
    ax.set_title('Modelos Vencedores — Contagem', fontweight='bold', color=C['azul_esc'])
    ax.set_facecolor(C['cinza_cla'])

    # Por grupo de sinal
    ax2 = axes[1]
    if 'Grupo_Sinal' in previsoes_df.columns:
        por_sinal = (previsoes_df
                     .groupby(['Grupo_Sinal', 'Melhor_Modelo'])
                     .size().unstack(fill_value=0))
        por_sinal.plot(kind='bar', ax=ax2,
                       color=[COR_MODELO.get(c, C['azul_med']) for c in por_sinal.columns],
                       edgecolor='white')
        ax2.set_title('Vencedores por Grupo de Sinal', fontweight='bold', color=C['azul_esc'])
        ax2.tick_params(axis='x', rotation=20)
        ax2.legend(title='Modelo')
        ax2.set_facecolor(C['cinza_cla'])

    plt.tight_layout()
    plt.show()

---
## 2. Galeria de Plots por Série

Para cada série são exibidos:
- **Cartão de métricas** (código, nome, grupo de sinal, melhor modelo, RMSE, MAPE, previsões)
- **Plot completo** gerado pelo experimento (histórico + previsão no holdout + resíduos)

In [ ]:
def _html_card(codigo, row=None):
    """Gera cartão HTML com metadados da série."""
    nome  = nomes_map.get(str(codigo), 'N/D')
    serie = series_validas.get(str(codigo))
    n_obs = len(serie) if serie is not None else '?'

    if row is not None:
        grupo  = row.get('Grupo_Sinal', 'N/D')
        melhor = row.get('Melhor_Modelo', 'N/D')
        rmse   = row.get('RMSE_Teste', None)
        mape   = row.get('MAPE_Teste_Pct', None)
        nov    = row.get('Prev_Nov_2025_fmt', row.get('Prev_Nov_2025', '-'))
        dez    = row.get('Prev_Dez_2025_fmt', row.get('Prev_Dez_2025', '-'))
        rmse_s = f'{rmse:,.0f}' if pd.notna(rmse) else '-'
        mape_s = f'{mape:.2f}%' if pd.notna(mape) else '-'
    else:
        grupo = melhor = rmse_s = mape_s = nov = dez = 'N/D'

    cor_mod = COR_MODELO.get(melhor, '#333')
    return f"""
    <div style="border:1px solid #BDC3C7; border-radius:8px; margin:20px 0 4px 0;
                font-family:monospace; font-size:13px; overflow:hidden">
      <div style="background:#1B3A5C; color:white; padding:8px 14px">
        <strong>Código {codigo}</strong>
        &nbsp;|&nbsp; {nome[:90]}
      </div>
      <div style="display:flex; padding:8px 14px; gap:30px; flex-wrap:wrap;
                  background:#F4F6F9">
        <span><b>Grupo sinal:</b> {grupo}</span>
        <span><b>Observações:</b> {n_obs}</span>
        <span><b>Melhor modelo:</b>
          <span style="color:{cor_mod};font-weight:bold">{melhor}</span></span>
        <span><b>RMSE (holdout):</b> {rmse_s}</span>
        <span><b>MAPE (holdout):</b> {mape_s}</span>
        <span><b>Prev. Nov/25:</b> {nov}</span>
        <span><b>Prev. Dez/25:</b> {dez}</span>
      </div>
    </div>
    """

print('Função _html_card definida.')

In [ ]:
# ── Galeria completa ──────────────────────────────────────────────────────────
# Itera pelas séries do CSV de previsões (quando disponível) ou pelas séries válidas

if previsoes_df is not None:
    iteracao = [
        (str(row.get('Codigo', row.get('codigo', ''))), row.to_dict())
        for _, row in previsoes_df.iterrows()
    ]
else:
    iteracao = [(k, None) for k in sorted(series_validas.keys())]

for codigo, row in iteracao:
    display(HTML(_html_card(codigo, row)))

    plot_path = PLOTS_DIR / f'serie_{codigo}.png'
    if plot_path.exists():
        display(Image(str(plot_path), width=950))
    else:
        display(HTML(
            f'<p style="color:#C0392B;padding:4px 14px">'  
            f'Plot não encontrado: {plot_path.name} '
            f'— execute o experimento para gerar os gráficos.</p>'
        ))

---
## 3. ACF/PACF por Série

Correlograma da parte de **treino** de cada série.
- **ACF** (autocorrelação): indica memória cumulativa (inclui efeitos indiretos).
- **PACF** (autocorrelação parcial): mede correlação direta em cada lag — base para seleção de janela.
- A linha tracejada indica a banda de confiança 95% (±1,96/√n).

In [ ]:
try:
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    _HAS_SM = True
except ImportError:
    _HAS_SM = False
    print('statsmodels não disponível — instale com: pip install statsmodels')

from utils.preprocessing import classificar_sinal

codigos_analise = [
    str(row.get('Codigo', row.get('codigo', '')))
    for _, row in previsoes_df.iterrows()
] if previsoes_df is not None else sorted(series_validas.keys())

for codigo in codigos_analise:
    serie = series_validas.get(codigo)
    if serie is None:
        continue

    nome  = nomes_map.get(codigo, 'N/D')
    vals  = serie.values.astype(float)
    n     = len(vals)
    n_hld = max(3, min(12, int(n * 0.20)))
    treino_vals = vals[:-n_hld]
    n_tr  = len(treino_vals)
    max_lags = min(20, n_tr // 2 - 1)

    if max_lags < 1:
        print(f'[{codigo}] treino muito curto para ACF/PACF')
        continue

    fig, axes = plt.subplots(1, 2, figsize=(13, 3))
    fig.suptitle(
        f'Código {codigo} — {nome[:70]}\n'
        f'(treino: {n_tr} obs. | holdout: {n_hld} obs.)',
        fontsize=9, color=C['azul_esc']
    )

    if _HAS_SM:
        plot_acf(treino_vals, ax=axes[0], lags=max_lags,
                 alpha=0.05, color=C['azul_med'],
                 title=f'ACF — lag máx. {max_lags}')
        plot_pacf(treino_vals, ax=axes[1], lags=max_lags,
                  alpha=0.05, method='ywm', color=C['azul_med'],
                  title=f'PACF — lag máx. {max_lags}')
    else:
        # Fallback manual
        xm  = treino_vals - treino_vals.mean()
        var = np.dot(xm, xm)
        acf = [np.dot(xm[:n_tr-k], xm[k:]) / var for k in range(max_lags + 1)]
        banda = 1.96 / np.sqrt(n_tr)
        for ax, vals_corr, title in zip(
            axes, [acf, acf], ['ACF (fallback)', 'PACF (fallback N/D)']
        ):
            ax.bar(range(len(vals_corr)), vals_corr, color=C['azul_med'], alpha=0.7)
            ax.axhline(banda,  color='gray', linestyle='--', linewidth=0.8)
            ax.axhline(-banda, color='gray', linestyle='--', linewidth=0.8)
            ax.axhline(0, color='black', linewidth=0.6)
            ax.set_title(title)

    for ax in axes:
        ax.set_facecolor(C['cinza_cla'])

    plt.tight_layout()
    plt.show()

---
## 4. Análise por Grupo de Sinal

In [ ]:
from utils.preprocessing import classificar_sinal

# Classifica cada série e monta tabela de resumo
info_series = []
for cod, serie in series_validas.items():
    grupo = classificar_sinal(serie)
    info_series.append({
        'Codigo'     : cod,
        'Nome'       : nomes_map.get(cod, 'N/D')[:60],
        'Grupo_Sinal': grupo,
        'N_obs'      : len(serie),
        'Media'      : serie.mean(),
        'Std'        : serie.std(),
        'Min'        : serie.min(),
        'Max'        : serie.max(),
    })

info_df = pd.DataFrame(info_series)

print('Contagem por grupo de sinal:')
print(info_df['Grupo_Sinal'].value_counts().to_string())
print()

# Estatísticas descritivas por grupo
resumo_grupo = (
    info_df.groupby('Grupo_Sinal')[['N_obs', 'Media', 'Std']]
    .agg(['mean', 'median', 'min', 'max'])
    .round(2)
)
display(resumo_grupo)

In [ ]:
# Plot: distribuição de comprimento das séries por grupo de sinal
grupos = info_df['Grupo_Sinal'].unique()
cores_grupo = {
    'apenas_positivos'      : C['azul_med'],
    'apenas_negativos'      : C['vermelho'],
    'positivos_e_negativos' : C['amarelo'],
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
contagem = info_df['Grupo_Sinal'].value_counts()
bars = ax.bar(contagem.index, contagem.values,
              color=[cores_grupo.get(g, C['azul_med']) for g in contagem.index],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, contagem.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1, str(val),
            ha='center', fontweight='bold')
ax.set_title('Séries por Grupo de Sinal', fontweight='bold', color=C['azul_esc'])
ax.tick_params(axis='x', rotation=15)
ax.set_facecolor(C['cinza_cla'])

ax2 = axes[1]
for grupo in grupos:
    dados = info_df[info_df['Grupo_Sinal'] == grupo]['N_obs']
    ax2.hist(dados, bins=10, alpha=0.7,
             color=cores_grupo.get(grupo, C['azul_med']), label=grupo)
ax2.set_title('Distribuição do Comprimento das Séries', fontweight='bold', color=C['azul_esc'])
ax2.set_xlabel('Número de observações')
ax2.set_ylabel('Frequência')
ax2.legend(fontsize=8)
ax2.set_facecolor(C['cinza_cla'])

plt.tight_layout()
plt.show()

In [ ]:
# Exibe as séries brutas agrupadas por tipo de sinal (sem escalar)
for grupo in sorted(grupos):
    sub_codigos = info_df[info_df['Grupo_Sinal'] == grupo]['Codigo'].tolist()
    n = len(sub_codigos)
    if n == 0:
        continue

    display(HTML(f'<h4 style="color:#1B3A5C">Grupo: {grupo} ({n} séries)</h4>'))

    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows),
                             squeeze=False)

    for idx, cod in enumerate(sub_codigos):
        ax  = axes[idx // ncols][idx % ncols]
        serie = series_validas[cod]
        nome  = nomes_map.get(cod, 'N/D')
        cor   = cores_grupo.get(grupo, C['azul_med'])

        ax.plot(serie.index.astype(str), serie.values, color=cor, linewidth=1.4)
        ax.set_title(f'{cod}\n{nome[:40]}', fontsize=7, color=C['azul_esc'])
        ax.tick_params(axis='x', rotation=60, labelsize=5)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(
                lambda x, _: f'R${x/1e6:.1f}Mi' if abs(x) >= 1e6 else f'R${x:,.0f}'
            )
        )
        ax.tick_params(axis='y', labelsize=6)
        ax.set_facecolor(C['cinza_cla'])

    # Oculta eixos extras
    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.tight_layout()
    plt.show()

---
## 5. Distribuição de Métricas por Modelo

In [ ]:
if metricas_df is None:
    print('Métricas não disponíveis.')
else:
    modelos = sorted(metricas_df['Modelo'].unique())

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, metrica in zip(axes, ['RMSE', 'MAE', 'MAPE']):
        if metrica not in metricas_df.columns:
            ax.set_visible(False)
            continue
        dados = [metricas_df[metricas_df['Modelo'] == m][metrica].dropna().values
                 for m in modelos]
        bp = ax.boxplot(dados, labels=modelos, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 2},
                        flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5})
        for patch, mod in zip(bp['boxes'], modelos):
            patch.set_facecolor(COR_MODELO.get(mod, C['azul_med']))
            patch.set_alpha(0.75)

        ax.set_title(f'Distribuição de {metrica} por Modelo',
                     fontsize=11, fontweight='bold', color=C['azul_esc'])
        ax.set_ylabel(metrica)
        ax.set_facecolor(C['cinza_cla'])
        if metrica in ('RMSE', 'MAE'):
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(
                    lambda x, _: f'R${x/1e6:.1f}Mi' if abs(x) >= 1e6 else f'R${x:,.0f}'
                )
            )

    plt.tight_layout()
    plt.show()

In [ ]:
if metricas_df is not None:
    # RMSE médio por modelo — barras com erro padrão
    resumo_mod = (
        metricas_df.groupby('Modelo')[['RMSE', 'MAE', 'MAPE']]
        .agg(['mean', 'std'])
        .round(2)
    )
    display(HTML('<h4>Estatísticas por Modelo (média ± desvio padrão)</h4>'))
    display(resumo_mod)

    # Melhor modelo por série (percentual de vitórias)
    if 'Selecionado' in metricas_df.columns:
        vitorias = (
            metricas_df[metricas_df['Selecionado']]
            ['Modelo'].value_counts(normalize=True) * 100
        ).round(1)
        print('\nTaxa de vitórias (% séries em que o modelo foi selecionado):')
        print(vitorias.to_string())

---
## 6. Análise Detalhada de Série Individual

Altere `CODIGO` para inspecionar qualquer série em detalhe.

In [ ]:
# ── Configure o código da série aqui ─────────────────────────────────────────
CODIGO = list(series_validas.keys())[0]   # Troque pelo código desejado
# Exemplo: CODIGO = '1112510101'
# ─────────────────────────────────────────────────────────────────────────────

CODIGO = str(CODIGO)
serie  = series_validas.get(CODIGO)

if serie is None:
    print(f'Código {CODIGO!r} não encontrado nas séries válidas.')
    print('Séries disponíveis:', list(series_validas.keys())[:10], '...')
else:
    nome = nomes_map.get(CODIGO, 'N/D')
    print(f'Código : {CODIGO}')
    print(f'Nome   : {nome}')
    print(f'Período: {serie.index.min()} → {serie.index.max()}')
    print(f'N obs  : {len(serie)}')
    print(f'Grupo  : {classificar_sinal(serie)}')
    print()
    print(serie.describe().to_string())

In [ ]:
if serie is not None:
    vals   = serie.values.astype(float)
    idx    = serie.index
    n      = len(vals)
    n_hld  = max(3, min(12, int(n * 0.20)))
    treino = vals[:-n_hld]
    teste  = vals[-n_hld:]
    idx_tr = idx[:-n_hld]
    idx_ts = idx[-n_hld:]

    fig, axes = plt.subplots(3, 1, figsize=(13, 10),
                             gridspec_kw={'height_ratios': [2.5, 1, 1]})

    # ── Painel 1: Série completa ──────────────────────────────────────────────
    ax = axes[0]
    ax.plot(idx_tr.astype(str), treino, color=C['azul_med'], linewidth=1.8,
            label='Treino', zorder=3)
    ax.plot(idx_ts.astype(str), teste, color=C['azul_esc'], linewidth=2.0,
            linestyle='--', label='Holdout (real)', zorder=4)
    ax.axvline(x=str(idx_ts[0]), color='gray', linewidth=1.0,
               linestyle='--', alpha=0.6, label='Início holdout')
    ax.set_title(f'{CODIGO} — {nome[:80]}', fontsize=10,
                 fontweight='bold', color=C['azul_esc'])
    ax.set_ylabel('Valor (R$)')
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f'R${x/1e6:.1f}Mi' if abs(x) >= 1e6 else f'R${x:,.0f}'
        )
    )
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.legend(fontsize=9)
    ax.set_facecolor(C['cinza_cla'])

    # ── Painel 2: ACF do treino ───────────────────────────────────────────────
    ax2 = axes[1]
    max_lags = min(20, len(treino) // 2 - 1)
    if _HAS_SM and max_lags >= 1:
        plot_acf(treino, ax=ax2, lags=max_lags, alpha=0.05, color=C['azul_med'],
                 title=f'ACF — Treino (lag máx. {max_lags})')
    else:
        ax2.set_title('ACF (statsmodels não disponível)')
    ax2.set_facecolor(C['cinza_cla'])

    # ── Painel 3: PACF do treino ──────────────────────────────────────────────
    ax3 = axes[2]
    if _HAS_SM and max_lags >= 1:
        plot_pacf(treino, ax=ax3, lags=max_lags, alpha=0.05,
                  method='ywm', color=C['azul_med'],
                  title=f'PACF — Treino (lag máx. {max_lags})')
    else:
        ax3.set_title('PACF (statsmodels não disponível)')
    ax3.set_facecolor(C['cinza_cla'])

    plt.tight_layout()
    plt.show()

    # Exibe o plot do experimento para comparação
    plot_path = PLOTS_DIR / f'serie_{CODIGO}.png'
    if plot_path.exists():
        display(HTML('<b>Plot completo gerado pelo experimento:</b>'))
        display(Image(str(plot_path), width=950))

    # Métricas da série selecionada
    if metricas_df is not None and 'Codigo' in metricas_df.columns:
        sub = metricas_df[metricas_df['Codigo'].astype(str) == CODIGO]
        if not sub.empty:
            display(HTML('<b>Métricas no holdout — todos os modelos:</b>'))
            cols_met = [c for c in ['Modelo', 'RMSE', 'MAE', 'MAPE', 'Selecionado']
                        if c in sub.columns]
            display(
                sub[cols_met]
                .sort_values('RMSE')
                .style
                .format({'RMSE': '{:,.2f}', 'MAE': '{:,.2f}', 'MAPE': '{:.2f}%'})
                .hide(axis='index')
            )